# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [153]:
!pip install -q \
    pinecone \
    panns-inference \
    datasets \
    librosa \
    python-dotenv

In [154]:
#!pip install librosa
#!pip install panns-inference

In [155]:
# !pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

In [156]:
!pip install -q panns-inference

In [157]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [158]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [159]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [160]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [161]:
import torch
from panns_inference import AudioTagging

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AudioTagging(
    checkpoint_path=None,
    device=device
)

print(f"Using device: {device}")

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1
Using device: cuda


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [162]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [163]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [164]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [165]:
index_name = "audio-search-demo"

In [166]:
# ==========================
# Pinecone Initialization
# ==========================

import time
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec

# --------------------------
# Load API Key from Colab Secrets
# --------------------------
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError(
        "PINECONE_API_KEY not found. "
        "Go to the 🔑 Secrets tab in Colab and create a secret named 'PINECONE_API_KEY'."
    )

# --------------------------
# Initialize Pinecone
# --------------------------
pc = Pinecone(api_key=PINECONE_API_KEY)

# --------------------------
# Index configuration
# --------------------------
index_name = "audio-search-demo"

spec = ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

# --------------------------
# Create index if it doesn't exist
# --------------------------
existing_indexes = pc.list_indexes().names()

if index_name not in existing_indexes:
    print(f"Creating index '{index_name}'...")

    pc.create_index(
        name=index_name,
        dimension=2048,
        metric="cosine",
        spec=spec
    )

    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

    print("Index created.")

else:
    print(f"Index '{index_name}' already exists.")

# --------------------------
# Connect to index
# --------------------------
index = pc.Index(index_name)

print("\nIndex statistics:")
print(index.describe_index_stats())

Creating index 'audio-search-demo'...
Index created.

Index statistics:
DescribeIndexStatsResponse(dimension=2048, total_vector_count=0, metric='cosine', namespaces=0)


# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [167]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/32 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=2048, total_vector_count=2000, metric='cosine', namespaces=1)

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [168]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [169]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [170]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

QueryResponse(matches=[ScoredVector(id='400', score=0.999629855, values=[]), ScoredVector(id='1667', score=0.842086434, values=[]), ScoredVector(id='1666', score=0.831264794, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Mon, 03 Aug 2026 12:52:40 GMT', 'content-type': 'application/json', 'content-length': '202', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '32', 'x-pinecone-request-latency-ms': '39', 'x-envoy-upstream-service-time': '40', 'x-pinecone-response-duration-ms': '42', 'grpc-status': '0', 'server': 'envoy'}))

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results.

In [171]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [172]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [173]:
find_similar_audios(1642)

Query Audio:


Result:


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [174]:
find_similar_audios(452)

Query Audio:


Result:


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [175]:
# Create the data folder if it doesn't exist
!mkdir -p data

# Download the sample audio
!wget -O data/miaow_16k.wav https://storage.googleapis.com/audioset/miaow_16k.wav

--2026-08-03 12:52:41--  https://storage.googleapis.com/audioset/miaow_16k.wav
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.20.207, 108.177.98.207, 74.125.135.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.20.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215546 (210K) [audio/x-wav]
Saving to: ‘data/miaow_16k.wav’

data/miaow_16k.wav  100%[===================>] 210.49K  --.-KB/s    in 0.002s  

2026-08-03 12:52:41 (88.5 MB/s) - ‘data/miaow_16k.wav’ saved [215546/215546]



We can load the audio into a Numpy array as follows:

In [176]:
import os

print(os.path.exists("data/miaow_16k.wav"))

True


In [177]:
import librosa
from IPython.display import Audio

a, sr = librosa.load("data/miaow_16k.wav", sr=44100)

print("Sampling rate:", sr)
print("Length:", len(a))

Audio(a, rate=sr)

Sampling rate: 44100
Length: 296843


Now we generate the embeddings for this audio and query the Pinecone index.

In [178]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [179]:
pc.delete_index(index_name)

# Load Music Dataset
# Generate Music Embeddings
# Create Pinecone Index
# Retrieve Similar Songs
# Build Recommendation Function
# User Examples

In [194]:
# ==========================================
# PART 2 - Music Recommendation RAG
# ==========================================

!pip install -q datasets
!pip install -q librosa

In [195]:
from datasets import load_dataset

dataset = load_dataset(
    "lewtun/music_genres_small",
    split="train"
)

print(dataset)

Dataset({
    features: ['audio', 'song_id', 'genre_id', 'genre'],
    num_rows: 1000
})


In [196]:
sample = dataset[0]

print(sample.keys())
print(sample)

dict_keys(['audio', 'song_id', 'genre_id', 'genre'])
{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7c085d0be9c0>, 'song_id': 11483, 'genre_id': 9, 'genre': 'International'}


In [197]:
from IPython.display import Audio

sample = dataset[0]

Audio(
    sample["audio"]["array"],
    rate=sample["audio"]["sampling_rate"]
)

In [198]:
sample = dataset[0]

print(sample["genre"])
print(sample["song_id"])

audio = sample["audio"]

print(audio)

International
11483


In [200]:
audio = dataset[0]["audio"]

print(type(audio))

<class 'datasets.features._torchcodec.AudioDecoder'>


In [201]:
audio = dataset[0]["audio"]

waveform = audio["array"]
sr = audio["sampling_rate"]

print(waveform.shape)
print(sr)

(1322816,)
44100


In [202]:
# ==========================================
# PART 3 - Audio Embeddings with CLAP
# ==========================================
# Install CLAP dependencies

!pip install -q transformers librosa soundfile

In [203]:
import torch
from transformers import AutoProcessor, ClapModel

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    "laion/clap-htsat-unfused"
)

model = ClapModel.from_pretrained(
    "laion/clap-htsat-unfused"
).to(device)

print("Device:", device)
print("CLAP model loaded!")

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

Device: cuda
CLAP model loaded!


In [204]:
with torch.no_grad():
    output = model.get_audio_features(**inputs)

print(type(output))
print(output)

<class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
BaseModelOutputWithPooling(last_hidden_state=tensor([[[[-0.8974, -0.9430, -0.9514,  ..., -0.0707, -0.4519, -1.6304],
          [-0.4737, -0.6979, -0.3219,  ..., -0.5023, -1.1605, -1.3577]],

         [[-0.3714, -1.1131, -0.7548,  ...,  0.5536,  0.3236, -1.2200],
          [ 0.3195, -0.2701, -0.8044,  ...,  0.2373, -0.2006, -1.0769]],

         [[-0.5627, -0.2094, -0.5543,  ..., -0.1841,  0.0625, -0.3589],
          [-0.2121, -0.8723, -0.9856,  ..., -0.2744, -0.5207, -0.1949]],

         ...,

         [[ 0.4101,  0.9184,  0.4720,  ..., -0.1497,  0.0554, -1.4838],
          [ 0.4293,  0.6072,  0.0712,  ...,  0.4147,  0.3089, -0.8663]],

         [[ 0.3167, -0.4932, -0.6880,  ..., -0.2611, -0.1438,  0.0296],
          [-0.0117,  0.0782, -0.0158,  ..., -0.0307,  0.4652, -0.1389]],

         [[-0.1503,  0.4292,  1.0226,  ...,  0.8183,  1.4629,  0.0586],
          [-0.0604, -0.7714,  0.0596,  ..., -0.0311,  0.2252, -0.1831]]

In [206]:
sample = dataset[0]

print(type(sample["audio"]))
print(sample["audio"])

<class 'datasets.features._torchcodec.AudioDecoder'>


In [207]:
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]

print(type(audio))
print(audio.shape)
print(sampling_rate)

<class 'numpy.ndarray'>
(1322816,)
44100


In [208]:
import librosa
import numpy as np
import torch

# Load one sample
sample = dataset[0]

# Get waveform
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]

# Convert to numpy float32
audio = np.asarray(audio, dtype=np.float32)

# Resample to 48 kHz (required by CLAP)
if sampling_rate != 48000:
    audio = librosa.resample(
        audio,
        orig_sr=sampling_rate,
        target_sr=48000
    )
    sampling_rate = 48000

# Process audio
inputs = processor(
    audio=[audio],   # <-- IMPORTANT: pass a LIST
    sampling_rate=sampling_rate,
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    output = model.get_audio_features(**inputs)

print(type(output))
print(output)

<class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
BaseModelOutputWithPooling(last_hidden_state=tensor([[[[-0.1835, -0.3486,  0.9123,  ..., -0.1629, -1.0779, -0.3030],
          [-0.3569, -0.3192, -0.3274,  ..., -0.2879, -0.8229,  0.1050]],

         [[ 1.1910,  0.9972,  1.0033,  ...,  0.3478, -1.0870, -1.1329],
          [ 0.4218, -0.1888,  0.1772,  ...,  0.4030,  0.1270,  0.4531]],

         [[-0.2488, -0.0688, -0.5473,  ..., -0.5819, -0.6834, -0.7298],
          [-0.1388, -0.8265, -0.6035,  ..., -0.2386, -0.9422, -0.9967]],

         ...,

         [[-1.0843, -1.2516, -0.4961,  ..., -0.8019,  0.5239,  1.2840],
          [ 0.3684, -0.3610, -0.2722,  ...,  0.3310,  0.2653,  0.5676]],

         [[-0.0948, -0.1736, -0.2512,  ..., -0.3484,  0.5825,  1.0850],
          [-0.1349, -0.1429, -0.2116,  ..., -0.0878, -0.4031, -0.7520]],

         [[-0.8457, -1.3927, -1.3348,  ..., -1.0880, -0.1192, -0.4446],
          [-0.0477, -1.2063, -0.8291,  ..., -0.0664, -0.7474, -0.7642]]

In [209]:
# Extract the embedding vector

embedding = output.pooler_output.squeeze().cpu().numpy()

print("Embedding shape:", embedding.shape)
print(embedding[:10])

Embedding shape: (512,)
[-0.01418475 -0.03476574  0.05049467  0.09130318  0.0210285   0.04131879
  0.04866057 -0.0096634  -0.0156824  -0.013712  ]


In [210]:
# ==========================================
# Create Pinecone Index for Music
# ==========================================

import time
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "audio-music-clap"

spec = ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=512,      # CLAP embedding size
        metric="cosine",
        spec=spec
    )

    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=512, total_vector_count=100, metric='cosine', namespaces=1)


In [211]:
# ==========================================
# Create Pinecone Index for Audio Embeddings
# ==========================================

import time
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata

# Load API key
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "audio-music-clap"

spec = ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=512,
        metric="cosine",
        spec=spec
    )

    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

print("Index ready!")

index = pc.Index(index_name)

print(index.describe_index_stats())

Index ready!
DescribeIndexStatsResponse(dimension=512, total_vector_count=100, metric='cosine', namespaces=1)


In [212]:
from tqdm import tqdm
import librosa
import numpy as np

records = []

N = 100

for i in tqdm(range(N)):

    sample = dataset[i]

    # Load audio
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Convert to float32
    audio = np.asarray(audio, dtype=np.float32)

    # Resample to 48kHz for CLAP
    if sr != 48000:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=48000
        )
        sr = 48000

    # Generate embedding
    inputs = processor(
        audio=audio,
        sampling_rate=sr,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.get_audio_features(**inputs)

    embedding = output.pooler_output.squeeze().cpu().numpy().tolist()

    metadata = {
      "genre": sample["genre"],
      "song_id": int(sample["song_id"]),
      "dataset_index": i
    }

    records.append(
        (
            str(i),
            embedding,
            metadata
        )
    )

# Upload everything
index.upsert(vectors=records)

print(f"\nUploaded {len(records)} songs!")

print(index.describe_index_stats())

100%|██████████| 100/100 [00:17<00:00,  5.61it/s]



Uploaded 100 songs!
DescribeIndexStatsResponse(dimension=512, total_vector_count=100, metric='cosine', namespaces=1)


In [213]:
# ==========================================
# Semantic Music Search with CLAP
# ==========================================

import torch
from IPython.display import Audio, display

def recommend_music(query, top_k=5):

    # Encode the text query with CLAP
    inputs = processor(
        text=[query],
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.get_text_features(**inputs)

    query_embedding = output.pooler_output.squeeze().cpu().numpy().tolist()

    # Search Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    print("=" * 80)
    print("USER QUERY:")
    print(query)
    print("=" * 80)

    # Display recommendations
    for i, match in enumerate(results["matches"], start=1):

        print(f"\nRecommendation #{i}")
        print(f"Similarity Score: {match['score']:.3f}")
        print(f"Genre: {match['metadata']['genre']}")
        print(f"Song ID: {match['metadata']['song_id']}")

        # Retrieve original audio
        idx = match["metadata"]["dataset_index"]

        print("\n▶ Playing recommendation:")

        display(
            Audio(
                dataset[idx]["audio"]["array"],
                rate=dataset[idx]["audio"]["sampling_rate"]
            )
        )

        print("-" * 80)

In [ ]:
recommend_music("Relaxing piano music")

In [ ]:
recommend_music(
    "Energetic rock music for working out"
)

In [ ]:
recommend_music("I want relaxing piano music")